<a href="https://colab.research.google.com/github/davidlealo/practicos_sisrec_2026/blob/main/practico04/04_content_based_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctico Content-based (Texto)

**Profesor:** Denis Parra

**Alumno:** `David Leal`


En este proyecto trabajaremos con un modelo de recomendacion de libros de la página [Goodreads](http://www.goodreads.com). El modelo de recomendación de libros es un recomendador basado en contenido, donde se utilizan modelos de lenguage BERT y BERT-large para el cálculo de embeddings de los libros y luego similaridades de ítems. Luego, dependiendo de los libros con los que el usuario ha interactuado, se recomiendan los ítems más similares.

In [ ]:
import numpy as np
import json
import requests
import heapq
import math
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA
from io import BytesIO
import pickle
import pandas as pd
import time

Descargamos datos que vienen previamente calculados:
- transacciones/interacciones de cada usuario
- transaciones para evaluar el modelo
- embeddings de descripciones calculados con BERT  
- embeddings de descripciones calculados con BERT-large
- datos de libros con información de titulo, descripcion, año de publicacion, entre otros.

In [ ]:
!wget https://www.dropbox.com/s/57tel5zqopkssrh/books.csv?dl=0 -O books.csv
!wget https://www.dropbox.com/s/zpnnoy1i8ljf9fg/goodreads_bert_embeddings.npy?dl=0 -O goodreads_bert_embeddings.npy
!wget https://www.dropbox.com/s/a8hcc9w30y7r3jl/goodreads_bert_large_embeddings.npy?dl=0 -O goodreads_bert_large_embeddings.npy
!wget https://www.dropbox.com/s/dqeqpsr0vdvmcy0/goodreads_past_interactions.json?dl=0 -O goodreads_past_interactions.json
!wget https://www.dropbox.com/s/rjtzhmb2zbpp30q/goodreads_test_interactions.json?dl=0 -O goodreads_test_interactions.json

--2026-03-30 17:59:28--  https://www.dropbox.com/s/57tel5zqopkssrh/books.csv?dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/5s6xrfnu17yi34sfhmskb/books.csv?rlkey=ymzokbyqw3qq2bq5okfao9w1z&dl=0 [following]
--2026-03-30 17:59:28--  https://www.dropbox.com/scl/fi/5s6xrfnu17yi34sfhmskb/books.csv?rlkey=ymzokbyqw3qq2bq5okfao9w1z&dl=0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucbbe78d71b255a0b109d63fa742.dl.dropboxusercontent.com/cd/0/inline/C9pJhWdJRnaVxrczWQk0LcVbMMzH6gCVXnJFA4lWLZgNSXY2JeiSCdDgr4AP5Ndu6MGesNDLlLr66LUENXE-1_bU-ZM22LvUAM1DU_R089yinFBXH9vfSgEi3GKMxxBaTBXASLZcxbdiBjdgA0QElAhr/file# [following]
--2026-03-30 17:59:28--  https://ucbbe78d71b255a0b109d63fa742.dl.dropboxusercontent.com/cd/0/in

# Cargar datos adicionales

In [ ]:
df_books = pd.read_csv('books.csv', sep=',')
df_books.head()

,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,book_desc
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,Winning will make you famous. Losing means cer...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,Harry Potter's life is miserable. His parents ...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,About three things I was absolutely positive.F...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,The unforgettable novel of a childhood in a sl...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,Alternate Cover Edition ISBN: 0743273567 (ISBN...


In [ ]:
df_books.columns

Index(['book_id', 'goodreads_book_id', 'best_book_id', 'work_id',
       'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year',
       'original_title', 'title', 'language_code', 'average_rating',
       'ratings_count', 'work_ratings_count', 'work_text_reviews_count',
       'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5',
       'image_url', 'small_image_url', 'book_desc'],
      dtype='object')

In [ ]:
print("# ítems   --> ", df_books["book_id"].nunique())

# ítems   -->  4287


In [ ]:
# diccionario con id del usuario y id de libros con los que ha interactuado en el pasado
with open('goodreads_past_interactions.json') as f:
    user_interactions = json.load(f)

# diccionario con id del usuario y id de libros para testear el modelo
with open('goodreads_test_interactions.json') as f:
    user_interactions_test = json.load(f)


In [ ]:
# dict index 2 book id and vice-versa for recommendation
idx2bookid = {i: id_ for i, id_ in enumerate(df_books.book_id)}
bookid2idx = {id_:i for i, id_ in enumerate(df_books.book_id)}

# Cargar características pre-entrenadas: BERT y BERT-large

En esta sección se trabajará con modelos pre-entrenados de modelos de lenguage BERT y BERT-large que convierten texto a embeddings.

Bidirectional Encoder Representations from Transformers (BERT) es una técnica de NLP (Natural Language Processing) desarrollada por Google y publicada en 2018 por Jacob Devlin.

Actualmente Google utiliza BERT para entender las consultas de los usuarios en su buscador.

Tiene dos versiones:
- **BERT:** 12 capas, 12 cabezales de atencion y 110 millones de parámetros. Genera vectores de 768 dimensiones
- **BERT-large:** 24 capas, 16 cabezales de atencion y 340 millones de parámetros. Genera vectores de 1024 dimensiones

![BERT y BERT-large](http://jalammar.github.io/images/bert-base-bert-large.png)

![BERT y BERT-large arquitectura](http://jalammar.github.io/images/bert-base-bert-large-encoders.png)

En este caso los textos que utilizaremos son los títulos de los libros con su descripción y compararemos los resultados de recomendación con BERT y BERT-large. Para efectos de este trabajo los vectores de características ya fueron entrenados y guardados en archivos numpy. A continuación son cargados en memoria.

Para mayores detalles sobre el modelo de lenguaje BERT se recomienda revisar el siguiente artículo:
- [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/pdf/1810.04805.pdf)


In [ ]:
bert_featmat = np.load('goodreads_bert_embeddings.npy', allow_pickle=True)
bert_large_featmat = np.load('goodreads_bert_large_embeddings.npy', allow_pickle=True)

In [ ]:
bert_featmat.shape

(4287, 768)

In [ ]:
bert_large_featmat.shape

(4287, 1024)

### **Pregunta 1**

Considerando que haremos un recomendador basado en contenidos ¿Por qué el uso de modelos de lenguage es una buena elección para este tipo de problema?

**Respuesta 1:**

# Probamos con BERT y BERT-large reduciendo dimensionalidad con PCA-20

Una vez calculado (o cargado) los vectores característicos de cada libro a partir de su titulo y descripción, reducimos dimensionalidad. Probaremos con BERT y BERT-large para comparar los resultados de ambos en recomendación basada en contenido.


In [ ]:
# Project into a 20 PCA feature space
pca20_bert_featmat = PCA(n_components=20).fit_transform(bert_featmat)
pca20_bert_large_featmat = PCA(n_components=20).fit_transform(bert_large_featmat)

In [ ]:
pca20_bert_featmat.shape

(4287, 20)

In [ ]:
pca20_bert_large_featmat.shape

(4287, 20)

### **Pregunta 2**

Comente por qué se utiliza PCA para reducir la dimensión de cada vector característico. ¿Qué sucede con la pérdida de información en la reducción de dimensionalidad?

**Respuesta 2:**

# Similar document retrieval

En esta sección utilizaremos los vectores cargados para hacer un sistema de recuperación o búsqueda de información, para diferentes métricas de distancia.

Buscamos libros similares de acuerdo a la representación vectorial (BERT) de su título y descripción.


In [ ]:
# format results
pd.options.display.max_colwidth = 50 # ...máximo de columna

In [ ]:
# Find similar images by image id
def find_similar_books(embedding, query_id=None, metric='euclidean', topk=10):

    n = embedding.shape[0]

    if query_id is None:
        query_i = np.random.randint(n)
        query_id = idx2bookid[query_i]

    else:
        query_i = bookid2idx[query_id]


    distances = pairwise_distances(embedding[query_i].reshape(1,-1), embedding, metric=metric)
    heap = []
    for i in range(n):
        if len(heap) < topk:
            heapq.heappush(heap, (-distances[0][i], i))
        else:
            heapq.heappushpop(heap, (-distances[0][i], i))

    heap.sort(reverse=True)
    rec_ids = [idx2bookid[i] for _,i in heap]

    return rec_ids

## Usando BERT

In [ ]:
# libros similares al libro de id 41865 (Twilight) utilizando distancia euclideana. se puede cambiar a "cosine"
similar_books = find_similar_books(bert_featmat, query_id = 3, metric = 'euclidean', topk=10 )
similar_books

[3, 2908, 3115, 2303, 7334, 7235, 5721, 3510, 9552, 9696]

In [ ]:
df_books[df_books.book_id.isin(similar_books)][['book_id', 'original_title', 'book_desc', 'authors']]

,book_id,original_title,book_desc,authors
2,3,Twilight,About three things I was absolutely positive.F...,Stephenie Meyer
1423,2303,Bloody Bones,"In Laurell K. Hamilton's ""New York Times"" best...",Laurell K. Hamilton
1708,2908,"Severed Heads, Broken Hearts",Robyn Schneider's The Beginning of Everything ...,Robyn Schneider
1808,3115,A Hunger Like No Other,In New York Times and USA TODAY bestselling au...,Kresley Cole
1999,3510,"Cerulean Sins (Anita Blake, Vampire Hunter, #11)","Cerulean Sins, the eleventh entry in the hugel...",Laurell K. Hamilton
2903,5721,No Humans Involved,In her acclaimed Women of the Otherworld serie...,Kelley Armstrong
3423,7235,Afterburn,#1 New York Times bestselling author Sylvia Da...,Sylvia Day
3459,7334,Graffiti Moon,"Lucy is in love with Shadow, a mysterious graf...",Cath Crowley
4152,9552,The Last Werewolf,"Here is a powerful, definitive new version of ...",Glen Duncan
4198,9696,Perfect Shadow,Discover the origins of Durzo Blint in this or...,Brent Weeks


## Usando BERT reducidos con PCA

In [ ]:
# libros similares al libro de id 41865 (Twilight) utilizando distancia euclideana. se puede cambiar a "cosine"
similar_books = find_similar_books(pca20_bert_featmat, query_id = 3, metric = 'euclidean', topk=10 )
similar_books

[3, 1354, 2908, 476, 1886, 7235, 2041, 9552, 4754, 6011]

In [ ]:
df_books[df_books.book_id.isin(similar_books)][['book_id', 'original_title', 'book_desc', 'authors']]

,book_id,original_title,book_desc,authors
2,3,Twilight,About three things I was absolutely positive.F...,Stephenie Meyer
354,476,The World According to Garp,"This is the life and times of T. S. Garp, the ...",John Irving
919,1354,The Strange Case of Dr. Jekyll and Mr. Hyde an...,\r\r\r\r\r\r\n'He put the glass to his lips an...,"Robert Louis Stevenson, Robert Mighall"
1201,1886,Preacher Vol. 1: Gone to Texas,One of the most celebrated comics titles of th...,"Garth Ennis, Steve Dillon, Joe R. Lansdale"
1289,2041,Lisey's Story,Lisey Landon lost her husband Scott two years ...,Stephen King
1708,2908,"Severed Heads, Broken Hearts",Robyn Schneider's The Beginning of Everything ...,Robyn Schneider
2524,4754,Midnight Bayou,#1 New York Times bestselling author Nora Robe...,Nora Roberts
3009,6011,What Do You Care What Other People Think? Furt...,One of the greatest physicists of the twentiet...,Richard Feynman
3423,7235,Afterburn,#1 New York Times bestselling author Sylvia Da...,Sylvia Day
4152,9552,The Last Werewolf,"Here is a powerful, definitive new version of ...",Glen Duncan


## Usando BERT-large

In [ ]:
# libros similares al libro de id 41865 (Twilight) utilizando distancia euclideana. se puede cambiar a "cosine"
similar_books = find_similar_books(bert_large_featmat, query_id = 3, metric = 'euclidean', topk=10 )
similar_books

[3, 7818, 712, 5118, 1818, 7221, 2112, 7101, 8266, 7552]

In [ ]:
df_books[df_books.book_id.isin(similar_books)][['book_id', 'original_title', 'book_desc', 'authors']]

,book_id,original_title,book_desc,authors
2,3,Twilight,About three things I was absolutely positive.F...,Stephenie Meyer
527,712,"Drums of Autumn (Outlander, #4)",In this breathtaking novel—rich in history and...,Diana Gabaldon
1160,1818,One Foot in the Grave,"You can run from the grave, but you can’t hide...",Jeaniene Frost
1321,2112,P.S. I Still Love You,Lara Jean didn’t expect to really fall for Pet...,Jenny Han
2672,5118,Stars Above,The enchantment continues....The universe of t...,Marissa Meyer
3380,7101,Magic Shifts,In the latest Kate Daniels novel from #1 New Y...,Ilona Andrews
3421,7221,Eeny Meeny,"The ""dark, twisted, thought-provoking"" (#1 New...",M.J. Arlidge
3532,7552,"The Dirty Life: On Farming, Food, and Love","""This book is the story of the two love affair...",Kristin Kimball
3630,7818,Frostbitten,"Smart, sexy, supernatural—the men and women of...",Kelley Armstrong
3764,8266,Bloodlust,\r\r\r\r\r\r\nA new beginning...\r\r\r\r\r\r\n...,"L.J. Smith, Kevin Williamson, Julie Plec"


## Usando BERT-large reducidos con PCA

In [ ]:
# libros similares al libro de id 41865 (Twilight) utilizando distancia euclideana. se puede cambiar a "cosine"
similar_books = find_similar_books(pca20_bert_large_featmat, query_id = 3, metric = 'euclidean', topk=10 )
similar_books

[3, 1818, 6977, 3549, 3984, 7818, 7221, 2476, 2483, 3408]

In [ ]:
df_books[df_books.book_id.isin(similar_books)][['book_id', 'original_title', 'book_desc', 'authors']]

,book_id,original_title,book_desc,authors
2,3,Twilight,About three things I was absolutely positive.F...,Stephenie Meyer
1160,1818,One Foot in the Grave,"You can run from the grave, but you can’t hide...",Jeaniene Frost
1492,2476,The Winter Sea,"In the spring of 1708, an invading Jacobite fl...","Susanna Kearsley, Rosalyn Landor"
1496,2483,Rush,"Gabe, Jace, and Ash: three of the wealthiest, ...",Maya Banks
1944,3408,Hunting Ground,Anna Latham didn’t know how complicated life c...,Patricia Briggs
2019,3549,Fever,"Jace, Ash, and Gabe: three of the wealthiest, ...",Maya Banks
2199,3984,The Darkest Whisper,New York Times bestselling sensation Gena Show...,Gena Showalter
3344,6977,The Mortal Instrument Series: City of Bones / ...,The Shadowhunters—touched by angels and charge...,Cassandra Clare
3421,7221,Eeny Meeny,"The ""dark, twisted, thought-provoking"" (#1 New...",M.J. Arlidge
3630,7818,Frostbitten,"Smart, sexy, supernatural—the men and women of...",Kelley Armstrong


### Pregunta 3:
Comente los resultados obtenidos, en cuanto a modelo de lenguaje, reduccion de dimensionalidad y métrica de distancia utilizada.


**Respuesta 3:**

# Recomendaciones

In [ ]:
# format results
pd.options.display.max_colwidth = 50
#pd.set_option('display.max_colwidth', -1)

## Función para obtener recomendacion para cada usuario

In [ ]:
def recommend(embedding, user_id=None, topk=10, metric='cosine'):

    #print("user_id = ", user_id)

    user_id = str(user_id)

    #Calculate distance metrics
    trx = user_interactions[user_id]
    n = embedding.shape[0]
    distances = 1e9

    # recorremos transacciones pasadas del usuario
    for t in trx:
        query_i = bookid2idx[t]

        # recomendamos items más cercanos a items con los que interactuó el usuario
        distances = np.minimum(distances, pairwise_distances(
                embedding[query_i].reshape(1,-1), embedding, metric=metric).reshape(-1))

    #Rank items de menor a mayor distancia (nos quedamos con los topk)
    trx_set = set(trx)
    heap = []
    for i in range(n):
        if idx2bookid[i] in trx_set:
            continue
        if len(heap) < topk:
            heapq.heappush(heap, (-distances[i], i))
        else:
            heapq.heappushpop(heap, (-distances[i], i))
    heap.sort(reverse=True)

    # utilizamos un heap para extraer los items ordenados de menor a mayor distancia
    recommended_ids = [idx2bookid[i] for _,i in heap]

    # retornar los que el usuario no haya consumido
    filtered_recommended_ids = []

    return recommended_ids

## Generar recomendaciones para un usuario en particular

In [ ]:
# recomendación para el usuario id = 49299 , utilizando bert con reduccion de dimensionalidad a 20
user_id = '50101'
rec = recommend(pca20_bert_featmat, user_id=user_id, topk=15)
rec

[4509,
 5002,
 4376,
 964,
 2292,
 7937,
 5126,
 9473,
 5244,
 390,
 6219,
 7602,
 7913,
 6865,
 2796]

## Transacciones pasadas del usuario

In [ ]:
past_interactions = user_interactions[str(user_id)]

df_books[df_books['book_id'].isin(past_interactions)][['book_id', 'original_title', 'book_desc', 'authors']]


,book_id,original_title,book_desc,authors
2,3,Twilight,About three things I was absolutely positive.F...,Stephenie Meyer
5,6,The Fault in Our Stars,Despite the tumor-shrinking medical miracle th...,John Green
7,8,The Catcher in the Rye,The hero-narrator of The Catcher in the Rye is...,J.D. Salinger
8,10,Pride and Prejudice,«È cosa ormai risaputa che a uno scapolo in po...,Jane Austen
9,11,The Kite Runner,"“It may be unfair, but what happens in a few ...",Khaled Hosseini
...,...,...,...,...
2344,4323,El Deafo,"Starting at a new school is scary, even more s...","Cece Bell, David Lasky"
2804,5476,The Crossover,"""With a bolt of lightning on my kicks . . .The...",Kwame Alexander
3004,5992,Ball Four,Twentieth-anniversary edition of a baseball cl...,Jim Bouton
3703,8057,Absent in the Spring,Returning from a visit to her daughter in Iraq...,"Mary Westmacott, Agatha Christie"


## Información de recomendaciones

In [ ]:
df_books[df_books['book_id'].isin(rec)][['book_id', 'original_title', 'book_desc', 'authors']]

,book_id,original_title,book_desc,authors
293,390,The Strange Case of Dr Jekyll and Mr Hyde,"Un abogado, Gabriel John Utterson, investiga l...","Robert Louis Stevenson, Vladimir Nabokov, Merv..."
683,964,The Hobbit and The Lord of the Rings,لجزء الثالث من ملحمة جيه أر أر تولكين الرائعة ...,J.R.R. Tolkien
1417,2292,ساق البامبو [Saq al-Bambu],لماذا كان جلوسي تحت الشجرة يزعج أمي؟ أتراها كا...,"سعود السنعوسي, Saud Alsanousi"
1643,2796,The Lost Wife,"Praga, 1938. Lenka, una joven estudiante de ar...",Alyson Richman
2363,4376,شيكاجو,يقول الأستاذ جلال أمين عن هذه الرواية المتميزة...,"Alaa Al Aswany, علاء الأسواني"
2418,4509,2666,"A cuatro profesores de literatura, Pelletier, ...",Roberto Bolaño
2629,5002,في ديسمبر تنتهي كل الأحلام,"أنا مُكتئب , مُكتئب جداً ! .. وعادة لا تُصيبني...",أثير عبدالله النشمي
2676,5126,Travesuras de la niña mala,¿Cuál es el verdadero rostro del amor?Ricardo ...,Mario Vargas Llosa
2718,5244,Il barone rampante,"Cuando tenía doce años, Cosimo Piovasco, barón...","Italo Calvino, Archibald Colquhoun"
3092,6219,Hyperspace: A Scientific Odyssey Through Paral...,¿Hay otras dimensiones más allá de las de nues...,Michio Kaku


# Evaluación de las recomendaciones con interacciones de testing

In [ ]:
# Métricas de evaluación
# Obtenido de https://gist.github.com/bwhite/3726239

def precision_at_k(r, k):
    assert k >= 1
    r = np.asarray(r)[:k] != 0
    if r.size != k:
        raise ValueError('Relevance score length < k')
    return np.mean(r)

def average_precision(r):
    r = np.asarray(r) != 0
    out = [precision_at_k(r, k + 1) for k in range(r.size) if r[k]]
    if not out:
        return 0.
    return np.mean(out)

def mean_average_precision(rs):
    return np.mean([average_precision(r) for r in rs])

def dcg_at_k(r, k):
    r = np.asarray(r)[:k]
    if r.size:
        return np.sum(np.subtract(np.power(2, r), 1) / np.log2(np.arange(2, r.size + 2)))
    return 0.


def ndcg_at_k(r, k):
    idcg = dcg_at_k(sorted(r, reverse=True), k)

    if not idcg:
        return 0.
    return dcg_at_k(r, k) / idcg


## Evaluación de recomendación con BERT

In [ ]:
start = time.time()

mean_map = 0.
mean_ndcg = 0.

embeddings = pca20_bert_featmat
topk = 10

for i, u in enumerate(user_interactions_test.keys()):

    print(i, end= '\r')

    rec = recommend(bert_featmat, user_id = u, topk=topk)
    rel_vector = [np.isin(user_interactions_test[u], rec, assume_unique=True).astype(int)]
    mean_map += mean_average_precision(rel_vector)
    mean_ndcg += ndcg_at_k(rel_vector, topk)

mean_map /= len(user_interactions_test)
mean_ndcg /= len(user_interactions_test)

time_taken = time.time() - start


In [ ]:
print('MAP ',mean_map)
print('ndcg@10' ,mean_ndcg)
print('tiempo de ejecucion {0:.2f} segs'.format(time_taken))


MAP  0.03669047619047619
ndcg@10 0.13
tiempo de ejecucion 261.57 segs


## Evaluación de recomendación con BERT reducidos con PCA-20

In [ ]:
start = time.time()

mean_map = 0.
mean_ndcg = 0.

embeddings = pca20_bert_featmat
topk = 10

for i, u in enumerate(user_interactions_test.keys()):

    print(i, end= '\r')

    rec = recommend(pca20_bert_featmat, user_id = u, topk=topk)
    rel_vector = [np.isin(user_interactions_test[u], rec, assume_unique=True).astype(int)]
    mean_map += mean_average_precision(rel_vector)
    mean_ndcg += ndcg_at_k(rel_vector, topk)

mean_map /= len(user_interactions_test)
mean_ndcg /= len(user_interactions_test)

time_taken = time.time() - start


In [ ]:
print('MAP ',mean_map)
print('ndcg@10' ,mean_ndcg)
print('tiempo de ejecucion {0:.2f} segs'.format(time_taken))

MAP  0.00453968253968254
ndcg@10 0.03
tiempo de ejecucion 13.03 segs


## Evaluación de recomendación con BERT-large

In [ ]:
start = time.time()

mean_map = 0.
mean_ndcg = 0.

topk = 10

for i, u in enumerate(user_interactions_test.keys()):

    print(i, end= '\r')

    rec = recommend(bert_large_featmat, user_id = u, topk=topk)
    rel_vector = [np.isin(user_interactions_test[u], rec, assume_unique=True).astype(int)]
    mean_map += mean_average_precision(rel_vector)
    mean_ndcg += ndcg_at_k(rel_vector, topk)

mean_map /= len(user_interactions_test)
mean_ndcg /= len(user_interactions_test)

time_taken = time.time() - start

In [ ]:
print('MAP ',mean_map)
print('ndcg@10' ,mean_ndcg)
print('tiempo de ejecucion {0:.2f} segs'.format(time_taken))

MAP  0.060920634920634924
ndcg@10 0.24
tiempo de ejecucion 442.03 segs


## Evaluación de recomendación con BERT-large reducidos con PCA-20

In [ ]:
start = time.time()

mean_map = 0.
mean_ndcg = 0.

topk = 10

for i, u in enumerate(user_interactions_test.keys()):

    print(i, end= '\r')

    rec = recommend(pca20_bert_large_featmat, user_id = u, topk=topk)
    rel_vector = [np.isin(user_interactions_test[u], rec, assume_unique=True).astype(int)]
    mean_map += mean_average_precision(rel_vector)
    mean_ndcg += ndcg_at_k(rel_vector, topk)

mean_map /= len(user_interactions_test)
mean_ndcg /= len(user_interactions_test)

time_taken = time.time() - start


In [ ]:
print('MAP ',mean_map)
print('ndcg@10' ,mean_ndcg)
print('tiempo de ejecucion {0:.2f} segs'.format(time_taken))

MAP  0.026123015873015874
ndcg@10 0.07
tiempo de ejecucion 13.07 segs


### Pregunta 4:
- Comente los resultados en términos de tiempo de ejecución y métricas de ranking para los 4 modelos.

**Respuesta 4:**